In [12]:
import kagglehub
from pathlib import Path
from PIL import Image

from matplotlib import path
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Download latest version
path = kagglehub.dataset_download("shuvoalok/dawn-dataset")

print("Path to dataset files:", path)

root = Path(path)
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
image_paths = [p for p in root.rglob("*") if p.suffix.lower() in image_exts]

print(f"Found {len(image_paths)} images under: {root}")
if not image_paths:
    raise FileNotFoundError("No image files found in dataset path.")

Path to dataset files: /Users/mpersson/.cache/kagglehub/datasets/shuvoalok/dawn-dataset/versions/5
Found 1005 images under: /Users/mpersson/.cache/kagglehub/datasets/shuvoalok/dawn-dataset/versions/5


In [13]:
import hashlib

# Remove duplicate image files by file content (keep the first occurrence)
before = len(image_paths)
seen_hashes = {}
duplicates = []

for p in image_paths:
    if not p.exists() or not p.is_file():
        continue

    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    digest = h.hexdigest()

    if digest in seen_hashes:
        duplicates.append(p)
    else:
        seen_hashes[digest] = p

removed = 0
for p in duplicates:
    p.unlink()
    removed += 1

# Refresh image list after deletion
image_paths = [p for p in root.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}]

print(f"Images before: {before}")
print(f"Duplicates removed: {removed}")
print(f"Images after: {len(image_paths)}")

Images before: 1005
Duplicates removed: 0
Images after: 1005


In [15]:
import json
import pandas as pd

# Use existing dataset root if available
dataset_root = root if "root" in globals() else Path(path)

# Try to find COCO-style annotation JSON first
json_candidates = sorted(
    [p for p in dataset_root.rglob("*.json") if any(k in p.name.lower() for k in ["coco", "annot", "instances"])]
)

label_files = sorted([p for p in dataset_root.rglob("*.txt") if "label" in str(p.parent).lower()])

if not label_files:
    raise FileNotFoundError("No COCO JSON or YOLO label .txt files found in dataset.")

image_exts = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
image_dirs = [p for p in dataset_root.rglob("*") if p.is_dir() and "image" in p.name.lower()]
image_dir = image_dirs[0] if image_dirs else dataset_root

rows = []

flip_counter = 0
for lf in label_files:
    stem = lf.stem
    img_path = next((image_dir / f"{stem}{ext}" for ext in image_exts if (image_dir / f"{stem}{ext}").exists()), None)

    img_w, img_h = None, None
    if img_path is not None:
        with Image.open(img_path) as im:
            img_w, img_h = im.size

    with open(lf, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue

            cls_id, cx, cy, bw, bh = map(float, parts[:5])


            # flip horizontal for all images in list
            flip_list = [
                "dusttornado-038.jpg",
                "dusttornado-909.jpg",
                "dusttornado-937.jpg",
                "rain_storm-901.jpg",
                "rain_storm-932.jpg",
                "rain_storm-939.jpg",
                "rain_storm-985.jpg",
                "sand_storm_g2_-026.jpg",
                "sand_storm_g2_-029.jpg",
                "sand_storm_g2_-063.jpg",
                "sand_storm_g2_-068.jpg",
                "sand_storm_g2_-079.jpg",
                "sand_storm-042.jpg",
                "sand_storm-077.jpg",
                "sand_storm-100.jpg",
                "sand_storm-103.jpg",
                "sand_storm-114.jpg",
                "sand_storm-129.jpg",
                "sand_storm-133.jpg",
                "sand_storm-138.jpg",
                "sand_storm-145.jpg",
                "sand_storm-157.jpg",
                "sand_storm-164.jpg",
                "sand_storm-165.jpg",
                "sand_storm-236.jpg",
                "sand_storm-246.jpg",
                "sand_storm-348.jpg",
                "sand_storm-359.jpg",
                "sand_storm-901.jpg",
                "sand_storm-910.jpg",
                "sand_storm-911.jpg",
                "sand_storm-913.jpg",
                "sand_storm-923.jpg",
                "sand_storm-924.jpg",
                "sand_storm-928.jpg",
                "sand_storm-940.jpg",
                "sand_storm-953.jpg",
                "sand_storm-989.jpg",
                "sand_storm-999.jpg"
            ]

            # Convert YOLO (cx, cy, w, h) to top-left (x1, y1, w, h)
            is_normalized = max(cx, cy, bw, bh) <= 1.0
            if img_w and img_h and is_normalized:
                x1 = (cx - bw / 2) * img_w
                y1 = (cy - bh / 2) * img_h
                w_abs = bw * img_w
                h_abs = bh * img_h
            else:
                x1 = cx - bw / 2
                y1 = cy - bh / 2
                w_abs, h_abs = bw, bh

            # Flip ground truth horizontally if image is in flip_list
            img_name = img_path.name if img_path is not None else f"{stem}.jpg"
            print(f"Processing {img_name} - Flip: {'Yes' if img_name in flip_list else 'No'}")
            if img_name in flip_list:
                if img_w:
                    x1 = img_w - (x1 + w_abs)
                elif is_normalized:
                    x1 = 1.0 - (x1 + w_abs)
                flip_counter += 1

            rows.append(
                {
                    "file_name": img_path.name if img_path else f"{stem}.*",
                    "width": img_w,
                    "height": img_h,
                    "category_id": int(cls_id),
                    "category": str(int(cls_id)),
                    "bbox_x": x1,
                    "bbox_y": y1,
                    "bbox_w": w_abs,
                    "bbox_h": h_abs,
                    "label_file": str(lf),
                }
            )

gt_df = pd.DataFrame(rows)
print(f"Loaded YOLO ground truth from {len(label_files)} label files")
print(f"Annotations: {len(gt_df)} | Images: {gt_df['file_name'].nunique() if not gt_df.empty else 0}")
print(f"Flipped annotations: {flip_counter}")
display(gt_df.head())

Processing dusttornado-001.jpg - Flip: No
Processing dusttornado-001.jpg - Flip: No
Processing dusttornado-001.jpg - Flip: No
Processing dusttornado-001.jpg - Flip: No
Processing dusttornado-001.jpg - Flip: No
Processing dusttornado-001.jpg - Flip: No
Processing dusttornado-001.jpg - Flip: No
Processing dusttornado-001.jpg - Flip: No
Processing dusttornado-001.jpg - Flip: No
Processing dusttornado-001.jpg - Flip: No
Processing dusttornado-001.jpg - Flip: No
Processing dusttornado-001.jpg - Flip: No
Processing dusttornado-001.jpg - Flip: No
Processing dusttornado-001.jpg - Flip: No
Processing dusttornado-001.jpg - Flip: No
Processing dusttornado-001.jpg - Flip: No
Processing dusttornado-002.jpg - Flip: No
Processing dusttornado-002.jpg - Flip: No
Processing dusttornado-002.jpg - Flip: No
Processing dusttornado-002.jpg - Flip: No
Processing dusttornado-002.jpg - Flip: No
Processing dusttornado-002.jpg - Flip: No
Processing dusttornado-002.jpg - Flip: No
Processing dusttornado-002.jpg - F

,file_name,width,height,category_id,category,bbox_x,bbox_y,bbox_w,bbox_h,label_file
0,dusttornado-001.jpg,900.0,562.0,3,3,397.0,408.0,120.0,86.0,/Users/mpersson/.cache/kagglehub/datasets/shuv...
1,dusttornado-001.jpg,900.0,562.0,3,3,526.0,430.0,85.0,62.0,/Users/mpersson/.cache/kagglehub/datasets/shuv...
2,dusttornado-001.jpg,900.0,562.0,6,6,496.0,292.0,82.0,89.0,/Users/mpersson/.cache/kagglehub/datasets/shuv...
3,dusttornado-001.jpg,900.0,562.0,3,3,76.0,340.0,201.0,167.0,/Users/mpersson/.cache/kagglehub/datasets/shuv...
4,dusttornado-001.jpg,900.0,562.0,3,3,667.0,347.0,64.0,55.0,/Users/mpersson/.cache/kagglehub/datasets/shuv...


In [16]:
# Display every image with its ground-truth bounding boxes
for file_name, g in gt_df.groupby("file_name", sort=True):
    img_file = image_dir / file_name
    if not img_file.exists():
        continue

    with Image.open(img_file) as img:
        fig, ax = plt.subplots(figsize=(12, 8))
        ax.imshow(img)
        ax.set_title(f"{file_name} | boxes: {len(g)}")
        ax.axis("off")

        for _, r in g.iterrows():
            rect = patches.Rectangle(
                (r["bbox_x"], r["bbox_y"]),
                r["bbox_w"],
                r["bbox_h"],
                linewidth=1.5,
                edgecolor="lime",
                facecolor="none",
            )
            ax.add_patch(rect)
            ax.text(
                r["bbox_x"],
                max(0, r["bbox_y"] - 3),
                str(r["category_id"]),
                color="yellow",
                fontsize=8,
                bbox=dict(facecolor="black", alpha=0.5, pad=1),
            )

        output_dir = "./data/gt_visualizations_fixed"
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        fig.savefig(Path(output_dir) / file_name, bbox_inches="tight", dpi=150)
        plt.close(fig)